In [6]:
import os, sys, json
sys.path.append("../")
from own_utils.oss_new import OssUtil
from own_utils.sql_python_utils import *
from own_utils.H5_utils import H5_utils
import json
import numpy as np
from multiprocessing import Pool
import pandas as pd

In [ ]:
country = 'new'
environment = "testing"
bucket = "cv-develop"
table_buried_point = 'id_sdk_liveness_buried_points_202504' 
table_detection = 'id_sdk_liveness_detection'
oss = OssUtil(country = country, environment = environment, bucket = bucket)
connect = Connector(country = country, environment = environment, database = "cv")
h5 = H5_utils(connect = connect, oss = oss, table_buried_point = table_buried_point, table_detection = table_detection)

In [9]:
def mediapipe106_h5_file_analysis(id):
    try:
        h5_file_detail = oss.get_h5_file(oss_id= id).split("\n")
        client = None
        backend = None
        detector_loading = []
        pfld_106_loading = []
        detector_init = []
        pfld_106_init = []
        detector_prediction = []
        pfld_106_prediction = []
        overall_processing = []
        for line in h5_file_detail:
            if 'client_type' in line:
                client = line.split('client_type')[-1].strip()
            if 'Backend' in line:
                backend = line.split('Backend')[-1].strip() 
            if 'detector init' in line:
                detector_init.append(int(line.split('detector init')[-1].strip()))
            if '106_pfld init' in line:
                pfld_106_init.append(int(line.split('106_pfld init')[-1].strip()))
            if '106_pfld loading ' in line:
                pfld_106_loading.append(int(line.split('106_pfld loading')[-1].strip()))
            if 'detector loading' in line:
                detector_loading.append(int(line.split('detector loading')[-1].strip()))
            if 'dectector prediction' in line:
                detector_prediction.append(int(line.split('dectector prediction')[-1].strip()))
            if '106_plfd prediction' in line:
                pfld_106_prediction.append(int(line.split('106_plfd prediction')[-1].strip()))
            if 'overall processing' in line:
                overall_processing.append(int(line.split('overall processing')[-1].strip()))
        
        detector_loading = int(np.mean(detector_loading)) if detector_loading else 'not found'
        pfld_106_loading = int(np.mean(pfld_106_loading)) if pfld_106_loading else 'not found'
        detector_init = int(np.mean(detector_init)) if detector_init else 'not found'
        pfld_106_init = int(np.mean(pfld_106_init)) if pfld_106_init else 'not found'
        detector_prediction = int(np.mean(detector_prediction)) if detector_prediction else 'not found'
        pfld_106_prediction = int(np.mean(pfld_106_prediction)) if pfld_106_prediction else 'not found'
        overall_processing = int(np.mean(overall_processing)) if overall_processing else 'not found'
        
        return (client, backend, detector_loading, pfld_106_loading, detector_init, pfld_106_init, detector_prediction, pfld_106_prediction, overall_processing)
    except Exception:
        return None

In [10]:
def process_row(row_input):
    # Assuming 'row' is a dictionary-like object containing necessary data
    index, row, new_columns = row_input
    result = None    
    if isinstance(row['H5_file'], str) and row['H5_file'].endswith(".txt") and row['H5_file'] != "no H5 file":
        oss_id = row['H5_file']
        result = mediapipe106_h5_file_analysis(id = oss_id)
        if not result:
            result = ['no H5 file'] * len(new_columns) 
    else:
        result = ['no H5 file'] * len(new_columns)
        
    return (index, result)

def multiprocess_row(df, new_columns, number_worker = 4):
    df = df.reset_index(drop= True)
    tasks = [(index, row.to_dict(), new_columns) for index, row in df.iterrows()]

    # Set up multiprocessing
    with Pool(processes = number_worker) as pool:
        results = pool.map(process_row, tasks)

    # Update DataFrame with results
    for index, update in results:
        df.loc[index, new_columns] = update

    return df

In [11]:
query = f"""
SELECT *
FROM cv.id_sdk_liveness_result
WHERE user_id in ("zox586f5d", "zoxf02e03", "zox060769", "zoxa4926a", "zoxbee50a", "zox6f7eed")
"""
liveness_data = connect.query(query=query)

print(f"Shape : {liveness_data.shape}")
liveness_data.drop_duplicates(subset= ['liveness_id'], inplace= True, keep= 'last')
liveness_data['data'] =  liveness_data['data'].apply(lambda x: json.loads(x))
liveness_data['ext_info'] =  liveness_data['ext_info'].apply(lambda x: json.loads(x))
liveness_data['liveness_result_msg'] = liveness_data['ext_info'].apply(lambda x: x.get("errMsg", None))
liveness_data['liveness_result_code'] = liveness_data['ext_info'].apply(lambda x: x.get("code", None))
liveness_data['sdk_version'] = liveness_data['ext_info'].apply(lambda x: x.get("sdk_version", None))
liveness_data['system'] = liveness_data['ext_info'].apply(lambda x: x.get("system", None))
liveness_data['deviceInfo'] = liveness_data['ext_info'].apply(lambda x: x.get("deviceInfo", {}).get('source', None))
liveness_result_df = liveness_data[['liveness_id', "create_time", 'partner_id', 'user_id', 'liveness_result_msg', 'liveness_result_code', 'sdk_version', 'system', 'deviceInfo']]
liveness_count_df = liveness_result_df.groupby('user_id').size().reset_index(name = 'user_liveness_count')
liveness_result_df = liveness_result_df.merge(right= liveness_count_df, on= 'user_id', how= 'inner')
user_id_unique = str(tuple(liveness_result_df['user_id'].unique()))
liveness_id_unique = str(tuple(liveness_result_df['liveness_id'].unique()))
buried_detail_df = h5.get_buried_detail(liveness_id= liveness_id_unique)
df = liveness_result_df.merge(buried_detail_df, on = 'liveness_id', how = 'left')
df = df[df["deviceInfo"] == "android-app"].drop(columns = ["detector_type", "create_time", "user_liveness_count", "partner_id"]).reset_index(drop = True)
df = df.drop_duplicates(subset= ['liveness_id'], keep= 'last').reset_index(drop = True)
new_column = ['client', 'backend', 'detector_loading', "pfld_106_loading", "detector_init", "pfld_106_init", "detector_prediction", "pfld_106_prediction", "overall_processing"]
result = None
for index, row in df.iterrows():
    oss_id = row['H5_file']
    result = mediapipe106_h5_file_analysis(id = oss_id)
    df.loc[index, new_column] = result
df

Shape : (6, 12)
Getting buried detail................
Getting buried detail done!!!


,liveness_id,user_id,liveness_result_msg,liveness_result_code,sdk_version,system,deviceInfo,H5_file,client,backend,detector_loading,pfld_106_loading,detector_init,pfld_106_init,detector_prediction,pfld_106_prediction,overall_processing
0,144a847b-5aba-403b-8273-f85249fbbc81,zox01e56f,活体认证成功,200,2.2.1.1,h5,android-app,d41830e60fdb3eb98da5e9d4ed8e4f97.txt,None,None,not found,not found,not found,not found,not found,not found,not found
1,a06c8c9f-d96f-46c9-a8cb-5d4ab2aec435,zox3ac708,活体认证成功,200,2.2.1.1,h5,android-app,4d4647282dc43cc3a82214ed272df06d.txt,None,None,not found,not found,not found,not found,not found,not found,not found
2,5da2571c-1c0b-4867-878b-0a828818b9a1,zox1c44dd,活体认证成功,200,2.2.1.1,h5,android-app,067b156426e03f02a7d7b9fc9135b52b.txt,None,None,not found,not found,not found,not found,not found,not found,not found
3,d14112ce-a779-48ad-9dd5-f920c85cc5a8,zox4c23ef,活体认证成功,200,2.2.1.1,h5,android-app,c1aaca96acfb351d859508355505befa.txt,None,None,not found,not found,not found,not found,not found,not found,not found
4,71e741e2-3b92-42af-85f6-34ed4e264568,zox244e7f,活体认证成功,200,2.2.1.1,h5,android-app,4b2f24e4992933d993805c4487723512.txt,None,None,not found,not found,not found,not found,not found,not found,not found
5,7d54ebd6-cbcd-4eff-99c3-233a048368c5,zoxf9e115,活体认证成功,200,2.2.1.1,h5,android-app,34137448a10e39d7b56c58dac010f327.txt,None,None,not found,not found,not found,not found,not found,not found,not found


In [5]:
user = "zox01e56f"
query = f"""
SELECT *
FROM cv.mx_sdk_liveness_result
WHERE user_id = "{user}"
"""
liveness_result_data = connect.query(query=query)
print(liveness_result_data['ext_info'].to_list())
liveness_result_data

KeyboardInterrupt: 

In [7]:

query = f"""
SELECT *
FROM cv.id_sdk_liveness_buried_points_202504
WHERE user_id = "{user}"
"""
buried_data = connect.query(query=query)
print(buried_data['data'].to_list()[0])
print(buried_data['ext_info'].to_list()[-1])
print(buried_data['data'].to_list()[2])
buried_data

第0次活体。手机品牌：vivo，手机型号：V2247，系统版本：14，系统sdk版本：34，系统语言：zh，CPU架构：arm64-v8a，GPU渲染器：Adreno (TM) 610，GPU供应商：Qualcomm，GPU版本：OpenGL ES-CM 1.1，WebView版本：134.0.6998.135
{"log_file_path": "/app/logs/8ffa776ef1393ba8862b66df0f52020d.txt"}
打开网页耗时：7979


,id,request_params,transaction_id,action,liveness_id,data,create_time,update_time,ext_info,user_id,sdk_version
0,2849,None,7314ff2e520730858a9d5c8c001c651b,Log,,第0次活体。手机品牌：vivo，手机型号：V2247，系统版本：14，系统sdk版本：34，...,2025-04-01 16:46:29.797900,2025-04-01 16:46:29.797700,None,zox6f7eed,AAR-2.0.2
1,2850,None,f0b3a6d8b889387eaa7ff1cce809fa4d,Log,,拿到livenessId：05c6ce94-20de-4e7a-88f5-b51433b126bf,2025-04-01 16:46:31.066600,2025-04-01 16:46:31.066500,None,zox6f7eed,AAR-2.0.2
2,2851,None,26fe21fb4222361a8baaa53384d48804,网页打开成功,05c6ce94-20de-4e7a-88f5-b51433b126bf,打开网页耗时：7979,2025-04-01 16:46:37.280200,2025-04-01 16:46:37.280100,None,zox6f7eed,AAR-2.0.2
3,2855,None,eac6aeb6b1fd343ba74f710ba306b027,H5_log_load_URL,05c6ce94-20de-4e7a-88f5-b51433b126bf,加载活体H5URL成功,2025-04-01 16:46:37.503900,2025-04-01 16:46:37.503700,"{""partner_id"": ""90""}",zox6f7eed,2.2.1.1
4,2856,None,1e45b5ef9a473c5bb81b1977ddd3ed86,H5_log_user_device_system_info,05c6ce94-20de-4e7a-88f5-b51433b126bf,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:5...,2025-04-01 16:46:37.506300,2025-04-01 16:46:37.506100,"{""partner_id"": ""90""}",zox6f7eed,2.2.1.1
5,2857,None,89e3f63973f23b7581c672603e79a040,H5_log_detector_type,05c6ce94-20de-4e7a-88f5-b51433b126bf,mediapipe106,2025-04-01 16:46:37.508700,2025-04-01 16:46:37.508600,"{""partner_id"": ""90""}",zox6f7eed,2.2.1.1
6,2858,None,b1c46b3bbc123a6cb0beb9b06f37cb80,H5_log_device_system_info,05c6ce94-20de-4e7a-88f5-b51433b126bf,iosundefined,2025-04-01 16:46:37.571900,2025-04-01 16:46:37.571700,"{""partner_id"": ""90""}",zox6f7eed,2.2.1.1
7,2859,None,21e41896fc0134898f38c329068e1a7c,H5_log_live_Count,05c6ce94-20de-4e7a-88f5-b51433b126bf,进入活体次数：1,2025-04-01 16:46:37.580900,2025-04-01 16:46:37.580800,"{""partner_id"": ""90""}",zox6f7eed,2.2.1.1
8,2860,None,e63163f60f5e31d3b2e90837caac7fb0,H5_log_camera_SUCCESS,05c6ce94-20de-4e7a-88f5-b51433b126bf,相机打开成功,2025-04-01 16:46:37.878500,2025-04-01 16:46:37.878300,"{""partner_id"": ""90""}",zox6f7eed,2.2.1.1
9,2861,None,6f4c933f2b4831e8a4a0188d5b2943b2,H5_log_load_blur_model_START,05c6ce94-20de-4e7a-88f5-b51433b126bf,加载模糊模型,2025-04-01 16:46:38.073000,2025-04-01 16:46:38.072900,"{""partner_id"": ""90""}",zox6f7eed,2.2.1.1


In [8]:
print(oss.get_h5_file(oss_id= "e243c59203f33269a92bf43161423499.txt"))

进入活体次数：8
client_type pc-browser
selected_model mediapipe106
system_info mozilla/5.0 (macintosh; intel mac os x 10_15_7) applewebkit/537.36 (khtml, like gecko) chrome/134.0.0.0 safari/537.36
鉴权参数 partner-code:cv timestamp:1744101633898 token:fc571cad6976d62ef1817557faad549c userId:zoxxh2kia device:pc-browser
selected_model: mediapipe_detector_106
收集网络数据
network 4g
Adapter [object GPUAdapter]
Device [object GPUDevice]
Resource analysis [{"name":"https://testh5.liveness.goodproduct.tech/resource/mxHld/js/safari-nomodule-fix.js","type":"script","transferSizeKB":1,"durationMs":157},{"name":"https://testh5.liveness.goodproduct.tech/resource/mxHld/js/chunk-vendors.b891e2bf.js","type":"script","transferSizeKB":704,"durationMs":4125},{"name":"https://testh5.liveness.goodproduct.tech/resource/mxHld/js/app.38605135.js","type":"script","transferSizeKB":146,"durationMs":1777},{"name":"https://testh5.liveness.goodproduct.tech/resource/mxHld/css/app.7792c910.css","type":"link","transferSizeKB":7,"dur

In [9]:
# query = f"""
# SELECT *
# FROM cv.id_sdk_liveness_result
# WHERE create_time >= '2025-03-01'
# AND create_time < '2025-03-21'
# AND partner_id in (167)
# """
# liveness_data = connect.query(query=query)

# print(f"Shape : {liveness_data.shape}")
# liveness_data.drop_duplicates(subset= ['liveness_id'], inplace= True, keep= 'last')
# liveness_data['data'] =  liveness_data['data'].apply(lambda x: json.loads(x))
# liveness_data['ext_info'] =  liveness_data['ext_info'].apply(lambda x: json.loads(x))
# liveness_data['liveness_result_msg'] = liveness_data['ext_info'].apply(lambda x: x.get("errMsg", None))
# liveness_data['liveness_result_code'] = liveness_data['ext_info'].apply(lambda x: x.get("code", None))
# liveness_data['sdk_version'] = liveness_data['ext_info'].apply(lambda x: x.get("sdk_version", None))
# liveness_data['system'] = liveness_data['ext_info'].apply(lambda x: x.get("system", None))
# liveness_data['deviceInfo'] = liveness_data['ext_info'].apply(lambda x: x.get("deviceInfo", {}).get('source', None))
# liveness_result_df = liveness_data[['liveness_id', "create_time", 'partner_id', 'user_id', 'liveness_result_msg', 'liveness_result_code', 'sdk_version', 'system', 'deviceInfo']]
# liveness_count_df = liveness_result_df.groupby('user_id').size().reset_index(name = 'user_liveness_count')
# liveness_result_df = liveness_result_df.merge(right= liveness_count_df, on= 'user_id', how= 'inner')
# user_id_unique = str(tuple(liveness_result_df['user_id'].unique()))
# liveness_id_unique = str(tuple(liveness_result_df['liveness_id'].unique()))
# buried_detail_df = h5.get_buried_detail(liveness_id= liveness_id_unique)
# df = liveness_result_df.merge(buried_detail_df, on = 'liveness_id', how = 'left')
# df = df[df["deviceInfo"] == "android-app"].drop(columns = ["detector_type", "create_time", "user_liveness_count", "partner_id"]).reset_index(drop = True)
# df = df.drop_duplicates(subset= ['liveness_id'], keep= 'last').reset_index(drop = True)

In [10]:
df = pd.read_csv("./data/benchmark.csv")

In [11]:
df

,liveness_id,user_id,liveness_result_msg,liveness_result_code,sdk_version,system,deviceInfo,H5_file
0,9213c4fd-c829-46e7-9567-923ce4803cd1,99E954E9-9FAD-492B-B90D-A66EBE8A314B,DETECTSUCCESS-waktu habis,40003,2.2.0.14,h5,android-app,00272471392739e88ac13604d103b919.txt
1,68ad951c-ed33-43c9-8020-64ef68c6a24a,1169736,ACTIONDOWN-waktu habis,40003,2.2.0.14,h5,android-app,8c5ed90319a5373b82c49860e80da77c.txt
2,fbb452bc-b157-4b83-94c8-42089a1a3c4c,1169989,Deteksi liveness sukses,200,2.2.0.14,h5,android-app,5a56e20ca2f83350ad17d0c6ff7a6c69.txt
3,8b801033-5d18-4f0c-83a7-efe2a7ea9d5c,1170047,Deteksi liveness sukses,200,2.2.0.14,h5,android-app,7176dab2763b34f0941af38363ccd63c.txt
4,723744ef-b9c1-4490-834d-3ec5e47ec3f6,99E954E9-9FAD-492B-B90D-A66EBE8A314B,Deteksi liveness sukses,200,2.2.0.14,h5,android-app,eea35773d7a730f5b3cdcf3e36ab6541.txt
...,...,...,...,...,...,...,...,...
25981,ebb819ca-784a-4fe8-8b7d-9a5bc0e158f9,1373173,Deteksi liveness sukses,200,2.2.0.14,h5,android-app,c09c6d37eb3333e8a5209e75abce88b2.txt
25982,e37304ca-61fa-43f6-a1f4-2c03aeac589c,1346069,Deteksi liveness sukses,200,2.2.0.14,h5,android-app,cf2f67f4d87f3e959fc82706e8d9d9ef.txt
25983,954f8c11-f057-488e-82f2-5d35f6e9eda4,846367,FACENODEFINE-waktu habis,40003,2.2.0.14,h5,android-app,50d0cf15e1d03741bec2570f1070f1d1.txt
25984,795563e2-7f74-4a86-8f72-0054eb5f6667,1373294,Deteksi liveness sukses,200,2.2.0.14,h5,android-app,c1af16d38bff3d2fa5933623c221c3e1.txt


In [12]:
new_column = ['client', 'backend', 'detector_loading', "pfld_106_loading", "detector_init", "pfld_106_init", "detector_prediction", "pfld_106_prediction", "overall_processing"]
df = multiprocess_row(df = df, new_columns = new_column, number_worker = 8)


Process SpawnPoolWorker-3:
Process SpawnPoolWorker-1:
Traceback (most recent call last):
  File "/Users/sheejiawei/.pyenv/versions/3.10.14/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/sheejiawei/.pyenv/versions/3.10.14/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/sheejiawei/.pyenv/versions/3.10.14/lib/python3.10/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/sheejiawei/.pyenv/versions/3.10.14/lib/python3.10/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'process_row' on <module '__main__' (built-in)>
Traceback (most recent call last):
  File "/Users/sheejiawei/.pyenv/versions/3.10.14/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/sheejiawei/.pyenv/versions/3.10.14/lib/python3.10/multiprocessing/process.py",

KeyboardInterrupt: 

In [ ]:
df.to_csv("./data/processed_benchmark.csv", index= False)